# Palier équité — mesurer et corriger le biais secteur

**Le diagnostic, déjà posé dans `baseline.ipynb` et `m2_m3_texte.ipynb` :** à chaque palier (M0 à M3), le taux d'approbation entre secteur formel et informel viole largement la règle des 4/5 (ratio ≈ 0,3–0,4), et surtout, le modèle **refuse disproportionnellement les « vrais bons » du secteur informel** — des clients qui n'auraient *pas* fait défaut (`gt_defaut_true == 0`), mais que le score juge trop risqués à cause de leur secteur et de ce qui lui est corrélé. C'est cette seconde mesure, pas la première, qui constitue l'injustice à corriger : un écart d'approbation peut refléter un vrai écart de risque, un écart de refus des vrais bons ne le peut pas.

**Objectif de ce notebook.** Corriger ce biais sur le modèle M3 (le plus performant construit à ce jour), en comparant deux familles de techniques :

- **A. Seuils différenciés par groupe** (post-traitement) — on ne touche pas au modèle ; on choisit, par secteur, un seuil d'approbation qui égalise le taux de refus des vrais bons entre groupes. C'est une version simplifiée de l'*equal opportunity* (Hardt, Price & Srebro, 2016).
- **B. Repondération des données d'entraînement** (pré-traitement, Kamiran & Calders, 2012) — chaque exemple d'entraînement est pondéré pour neutraliser la corrélation secteur↔défaut *avant* l'entraînement, puis le modèle est ré-entraîné.

Les deux sont comparées sur le même audit avant/après, avec un contrôle qu'elles ne dégradent pas l'équilibre déjà bon par sexe.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from categorisation import construire_parts_categories
from scoring_utils import (
    appliquer_seuils_groupe,
    audit_equite,
    construire_pipeline,
    construire_variables_comportementales,
    evaluer,
    poids_reponderation,
    resumer_decision_equite,
    seuils_egalite_opportunite,
)

SEED = 42
TARGET = "defaut_90j"

DECLARATIF_NUM = ["age", "revenu_declare", "anciennete_mois"]
DECLARATIF_CAT = ["zone", "secteur", "region"]
COMPORTEMENTAL_NUM = [
    "nb_tx", "pct_debits", "inflow", "outflow", "net_flow",
    "mean_abs", "std_abs", "max_abs", "cv_abs",
    "nb_jours_actifs", "tx_par_jour", "ecart_revenu",
]

# C déjà retenu pour M3 dans m2_m3_texte.ipynb (recherche de C non refaite ici, voir Étape 2)
C_M3 = 0.01

## Étape 1 — Reconstruire le jeu de données M3

Mêmes fonctions partagées que `m2_m3_texte.ipynb` (`scoring_utils.py`, `categorisation.py`), même split (`SEED=42, test_size=0.20`) que les deux autres notebooks — donc les mêmes clients en test partout.

In [2]:
clients = pd.read_csv("clients_synth.csv")
tx = pd.read_csv("transactions_synth.csv")

comp = construire_variables_comportementales(tx, clients)
parts, _ = construire_parts_categories(tx)
PART_COLS = [c for c in parts.columns if c.startswith("PART_")]
texte_par_client = (tx.groupby("client_id").libelle
                    .apply(lambda s: " ".join(map(str, s)))
                    .rename("texte")
                    .reset_index())

df = (clients
      .merge(comp, on="client_id", how="left")
      .merge(parts, on="client_id", how="left")
      .merge(texte_par_client, on="client_id", how="left"))
df[COMPORTEMENTAL_NUM + PART_COLS] = df[COMPORTEMENTAL_NUM + PART_COLS].fillna(0)
df["texte"] = df["texte"].fillna("")

y = df[TARGET].values
colonnes_gt = [c for c in df.columns if c.startswith("gt_")]
X = df.drop(columns=colonnes_gt + [TARGET, "client_id"])

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
dfte = df.loc[Xte.index]

NUM_M3 = DECLARATIF_NUM + COMPORTEMENTAL_NUM + PART_COLS
print(f"train {len(Xtr)} | test {len(Xte)}")

train 1600 | test 400


## Étape 2 — Ré-entraîner M3 et rappeler l'audit d'équité de référence

On reprend le `C` déjà retenu dans `m2_m3_texte.ipynb` (pas de nouvelle recherche de `C` ici — ce n'est pas l'objet de ce notebook). C'est ce modèle, non corrigé, qui sert d'état de référence *avant correction*.

In [3]:
pipeline_m3 = construire_pipeline(NUM_M3, DECLARATIF_CAT, texte_col="texte", seed=SEED)
pipeline_m3.set_params(clf__C=C_M3)
pipeline_m3.fit(Xtr, ytr)

p3 = pipeline_m3.predict_proba(Xte)[:, 1]
resultat_reference = evaluer("M3 (référence)", yte, p3)

print("\naudit d'équité par secteur, AVANT correction :")
taux_ref, ratio_ref, refus_ref = audit_equite(dfte, p3, "secteur", gt_true_col="gt_defaut_true")

M3 (référence) | AUC 0.719 | Gini 0.439 | KS 0.345

audit d'équité par secteur, AVANT correction :
taux d'approbation par secteur : {'formel': 0.77, 'informel': 0.272}
ratio (règle des 4/5) : 0.35 -> disparate impact
refus des vrais bons par secteur : {'formel': 0.183, 'informel': 0.704}


## Étape 3 — Méthode A : seuils différenciés par groupe (post-traitement)

Le score `p3` ne change pas : on choisit seulement, par secteur, un seuil d'approbation différent qui égalise le taux de refus des vrais bons sur celui du groupe le mieux traité. **Les seuils sont calculés sur le train**, jamais sur le test (sinon fuite), puis appliqués tels quels au test.

In [4]:
p3_train = pipeline_m3.predict_proba(Xtr)[:, 1]
dftr = df.loc[Xtr.index]

seuils_secteur = seuils_egalite_opportunite(dftr, p3_train, "secteur", "gt_defaut_true")
print(f"seuils par secteur (calculés sur le train) : {seuils_secteur}\n")

approuve_a = appliquer_seuils_groupe(dfte, p3, "secteur", seuils_secteur)
print("audit d'équité par secteur, APRÈS méthode A (seuils par groupe) :")
taux_a, ratio_a, refus_a = resumer_decision_equite(dfte, approuve_a, "secteur", gt_true_col="gt_defaut_true")

seuils par secteur (calculés sur le train) : {'formel': np.float64(0.4437673019956063), 'informel': 0.6154000374429781}

audit d'équité par secteur, APRÈS méthode A (seuils par groupe) :
taux d'approbation par secteur : {'formel': 0.77, 'informel': 0.677}
ratio (règle des 4/5) : 0.88 -> pas de signal au seuil des 4/5
refus des vrais bons par secteur : {'formel': 0.183, 'informel': 0.29}


## Étape 4 — Méthode B : repondération des données d'entraînement (pré-traitement)

Ici, c'est le modèle lui-même qui change : chaque exemple du train reçoit un poids `w(secteur,y) = P(secteur)·P(y) / P(secteur,y)` (Kamiran & Calders, 2012), qui réduit le poids des combinaisons sur-représentées et augmente celui des combinaisons sous-représentées, neutralisant une partie de la corrélation secteur↔défaut *avant* l'entraînement. `class_weight` est désactivé (`None`) pour ne pas cumuler deux rééquilibrages différents avec ces poids.

In [5]:
poids_train = poids_reponderation(ytr, Xtr["secteur"])

pipeline_m3_repondere = construire_pipeline(NUM_M3, DECLARATIF_CAT, texte_col="texte", seed=SEED)
pipeline_m3_repondere.set_params(clf__C=C_M3, clf__class_weight=None)
pipeline_m3_repondere.fit(Xtr, ytr, clf__sample_weight=poids_train)

p3_repondere = pipeline_m3_repondere.predict_proba(Xte)[:, 1]
resultat_b = evaluer("M3 (repondéré)", yte, p3_repondere)

print("\naudit d'équité par secteur, APRÈS méthode B (repondération) :")
taux_b, ratio_b, refus_b = audit_equite(dfte, p3_repondere, "secteur", gt_true_col="gt_defaut_true")

M3 (repondéré) | AUC 0.699 | Gini 0.397 | KS 0.341

audit d'équité par secteur, APRÈS méthode B (repondération) :
taux d'approbation par secteur : {'formel': 0.574, 'informel': 0.438}
ratio (règle des 4/5) : 0.76 -> disparate impact
refus des vrais bons par secteur : {'formel': 0.384, 'informel': 0.543}


## Étape 5 — Tableau comparatif

**À retenir avant de lire le tableau : la méthode A ne change pas l'AUC.** Elle ne touche pas au score `p3`, seulement à la décision de coupure — donc l'AUC de « M3 + seuils par groupe » est *par construction* identique à celle de M3 référence. La méthode B, elle, change le modèle : son AUC peut (et va probablement) baisser — c'est le coût attendu de la repondération.

In [6]:
comparatif = pd.DataFrame([
    {"modele": "M3 référence", "auc_test": resultat_reference["auc"],
     "ratio_4_5_secteur": ratio_ref,
     "refus_vrais_bons_formel": refus_ref.get("formel"),
     "refus_vrais_bons_informel": refus_ref.get("informel")},
    {"modele": "M3 + seuils par groupe (A)", "auc_test": resultat_reference["auc"],
     "ratio_4_5_secteur": ratio_a,
     "refus_vrais_bons_formel": refus_a.get("formel"),
     "refus_vrais_bons_informel": refus_a.get("informel")},
    {"modele": "M3 + repondération (B)", "auc_test": resultat_b["auc"],
     "ratio_4_5_secteur": ratio_b,
     "refus_vrais_bons_formel": refus_b.get("formel"),
     "refus_vrais_bons_informel": refus_b.get("informel")},
])
comparatif.round(3)

,modele,auc_test,ratio_4_5_secteur,refus_vrais_bons_formel,refus_vrais_bons_informel
0,M3 référence,0.719,0.353,0.183,0.704
1,M3 + seuils par groupe (A),0.719,0.879,0.183,0.290
2,M3 + repondération (B),0.699,0.763,0.384,0.543


## Étape 6 — Contrôle de non-régression par sexe

`sexe` était déjà équilibré avant correction (ratio 4/5 proche de 1). Les deux corrections ciblent `secteur` seul : on vérifie qu'elles ne dégradent pas l'équilibre par sexe au passage.

In [7]:
print("sexe, référence :")
_ = audit_equite(dfte, p3, "sexe", gt_true_col="gt_defaut_true")

print("\nsexe, après méthode A :")
_ = resumer_decision_equite(dfte, approuve_a, "sexe", gt_true_col="gt_defaut_true")

print("\nsexe, après méthode B :")
_ = audit_equite(dfte, p3_repondere, "sexe", gt_true_col="gt_defaut_true")

sexe, référence :
taux d'approbation par sexe : {'F': 0.517, 'M': 0.481}
ratio (règle des 4/5) : 0.93 -> pas de signal au seuil des 4/5
refus des vrais bons par sexe : {'F': 0.457, 'M': 0.463}

sexe, après méthode A :
taux d'approbation par sexe : {'F': 0.739, 'M': 0.698}
ratio (règle des 4/5) : 0.94 -> pas de signal au seuil des 4/5
refus des vrais bons par sexe : {'F': 0.239, 'M': 0.241}

sexe, après méthode B :
taux d'approbation par sexe : {'F': 0.526, 'M': 0.471}
ratio (règle des 4/5) : 0.90 -> pas de signal au seuil des 4/5
refus des vrais bons par sexe : {'F': 0.457, 'M': 0.481}


## Lecture des résultats

**Les deux méthodes resserrent l'écart de refus des vrais bons par secteur**, mais pas de la même manière ni au même coût :

- **A (seuils par groupe)** corrige la décision finale sans toucher au score ni à l'AUC — la moins chère en pouvoir prédictif, mais la moins fondamentale : le modèle continue de produire un score plus défavorable au secteur informel, on compense seulement au moment de la coupure. Elle suppose aussi qu'on accepte, en production, d'appliquer des seuils différents selon le secteur du demandeur — un choix qui doit être assumé et documenté auprès du régulateur, pas seulement technique.
- **B (repondération)** agit plus en amont — le score lui-même change — au prix d'une perte d'AUC probable : le modèle est délibérément moins optimisé sur le critère de performance brute pour l'être davantage sur l'équité.

**Limites assumées.** La correction ne porte que sur `secteur` seul ; elle n'a pas été vérifiée sur l'intersection secteur×sexe. Les seuils de la méthode A viennent d'une recherche sur grille (pas d'une garantie théorique d'égalité exacte), et la méthode B ne recherche pas un nouveau `C` sous poids — un `C` ré-optimisé sous repondération pourrait donner un meilleur compromis équité/performance que celui retenu ici.

## Limite à ne pas passer sous silence — ce biais est en partie construit par le générateur

Le diagnostic ci-dessus (et dans `baseline.ipynb`/`m2_m3_texte.ipynb`) a été présenté comme une découverte empirique sur les données. Ce n'en est qu'une partie honnête : `Générateur_données.ipynb` **couple explicitement le secteur informel au risque, à deux niveaux différents**, et il faut distinguer les deux avant de tirer une conclusion sur le « désavantage » du secteur informel :

1. **Un vrai écart de risque simulé (pas un artefact).** `income_stab` a une moyenne plus haute pour le secteur formel (`0.6 * (secteur == "formel")`), et `distress` une moyenne plus haute pour le secteur informel (`0.5 * (secteur == "informel") + 0.2 * camf`) ; ces deux variables entrent dans `C_star`, le score de risque *réel* simulé (`gt_defaut_true`). Le secteur informel a donc, par construction du générateur, un risque réellement plus élevé en moyenne — ce n'est pas en soi un bug ni un biais à corriger, juste un choix de simulation qui reproduit une hétérogénéité de risque plausible entre secteurs.
2. **Un biais d'étiquette injecté en plus, lui clairement artificiel.** `BIAIS_INFORMEL_FLIP = 0.06` fait ensuite basculer à 1 (défaut observé) 6 % des clients du secteur informel qui étaient de *vrais bons* (`defaut_true == 0`) — un mécanisme de biais d'étiquette injecté délibérément, indépendant du risque réel, pour que le pipeline de détection/correction ait quelque chose à détecter et corriger.

**Ce que l'audit ci-dessus démontre réellement :** que le pipeline (mesure du disparate impact, distinction risque réel/refus des vrais bons via `gt_defaut_true`, correction par seuils différenciés ou repondération) **fonctionne comme attendu sur un biais injecté connu** — il le détecte, le quantifie, et les deux méthodes de correction le resserrent bien. C'est une validation de la mécanique du pipeline, pas la preuve d'un désavantage structurel réel du secteur informel dans le monde réel : sur des données réelles, on ne connaît ni `gt_defaut_true` ni la part du biais qui serait due à un artefact d'étiquetage plutôt qu'à un vrai écart de risque ou à des variables omises corrélées au secteur. Toute transposition de cette conclusion à un cas réel devrait être revalidée sur un jeu de données réel avec une vérité-terrain indépendante (ou, à défaut, une analyse de sensibilité sur l'hypothèse de biais d'étiquette) avant d'informer une décision réglementaire ou commerciale.